# 01 — Ingest: NYC Yellow Taxi 2024

**Purpose:** Load all 12 months of Yellow Taxi parquet files into PostgreSQL `trips_raw` table, and load the zone lookup CSV into `taxi_zones`.

**Input:** `data/raw/yellow_tripdata_2024-01.parquet` through `...12.parquet`  
**Output:** `trips_raw` table (~36M rows), `taxi_zones` table (265 rows)

---

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from pathlib import Path
import time

# ─── CONFIG — update your password here ───────────────────────────────────────
DB_PASSWORD = 'YOUR_PASSWORD'   # <── change this
DB_USER     = 'postgres'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'taxi_db'
# ──────────────────────────────────────────────────────────────────────────────

# Paths — adjust if your folder structure differs
DATA_RAW_DIR       = Path('../data/raw')
ZONE_LOOKUP_PATH   = Path('../data/taxi_zone_lookup.csv')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Quick connection test
with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('✅ Connected to PostgreSQL:', result.scalar())

## 1. Load Zone Lookup → `taxi_zones`

265 rows mapping LocationID → Borough, Zone, service_zone.

In [ ]:
zones_df = pd.read_csv(ZONE_LOOKUP_PATH)

print(f'Shape:   {zones_df.shape}')
print(f'Columns: {zones_df.columns.tolist()}')
print()
zones_df.head()

In [ ]:
# Load to PostgreSQL (replace resets the table cleanly)
zones_df.to_sql('taxi_zones', con=engine, if_exists='replace', index=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'✅ taxi_zones loaded: {count} rows')

## 2. Load 12 Months of Trip Data → `trips_raw`

Each month is ~2–4M rows. We insert in chunks of 100k rows to keep memory usage low.

In [ ]:
# Preview what files we have
parquet_files = sorted(DATA_RAW_DIR.glob('yellow_tripdata_2024-*.parquet'))
print(f'Found {len(parquet_files)} parquet files:')
for f in parquet_files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f'  {f.name}  ({size_mb:.0f} MB)')

if len(parquet_files) != 12:
    print(f'⚠️  Expected 12 files, found {len(parquet_files)}. Check DATA_RAW_DIR path.')

In [ ]:
CHUNK_SIZE  = 100_000   # rows per DB insert batch
total_rows  = 0
start_time  = time.time()

for i, parquet_file in enumerate(parquet_files, start=1):
    t0 = time.time()
    df = pd.read_parquet(parquet_file)
    file_rows = len(df)

    # First file creates the table; subsequent files append
    mode = 'replace' if i == 1 else 'append'

    df.to_sql(
        'trips_raw',
        con=engine,
        if_exists=mode,
        index=False,
        chunksize=CHUNK_SIZE,
        method='multi'
    )

    total_rows += file_rows
    elapsed = time.time() - t0
    print(f'[{i:02d}/12] {parquet_file.name}  |  {file_rows:>8,} rows  |  {elapsed:.0f}s  |  running total: {total_rows:,}')

total_elapsed = time.time() - start_time
print(f'\n🎉 Done! {total_rows:,} rows loaded in {total_elapsed/60:.1f} min')

## 3. Verify in PostgreSQL

In [ ]:
with engine.connect() as conn:
    trips_count = conn.execute(text('SELECT COUNT(*) FROM trips_raw')).scalar()
    zones_count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'trips_raw  → {trips_count:,} rows  (expected ~36M)')
print(f'taxi_zones → {zones_count:,} rows  (expected 265)')

# Flag mismatch
if trips_count < 30_000_000:
    print('⚠️  trips_raw row count looks low — check if all 12 files loaded.')
else:
    print('✅ Row count looks good!')

In [ ]:
# Date range sanity check
date_check = pd.read_sql_query('''
    SELECT
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT DATE_TRUNC('month', tpep_pickup_datetime)) AS months_present
    FROM trips_raw
''', con=engine)

print('Date range check:')
date_check

In [ ]:
# Quick preview of raw data
sample = pd.read_sql_query('SELECT * FROM trips_raw LIMIT 5', con=engine)
print('Sample rows from trips_raw:')
sample

## ✅ Ingest Complete

| Table | Rows | Status |
|-------|------|--------|
| `trips_raw` | ~36M | ✅ Loaded |
| `taxi_zones` | 265 | ✅ Loaded |

**Next step:** Run `sql/02_clean.sql` in pgAdmin or psql to create the cleaned `trips_clean` table.